<a href="https://colab.research.google.com/github/VladShajdulin/OTUS/blob/main/home_work_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
! pip install evaluate
! git clone https://github.com/RussianNLP/RuCoLA

fatal: destination path 'RuCoLA' already exists and is not an empty directory.


In [29]:
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
from datasets import Dataset
from sklearn.model_selection import train_test_split
import evaluate

In [30]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else 'cpu'
print('Device', device)

Device cpu


In [31]:
train_df = pd.read_csv('/content/RuCoLA/data/in_domain_train.csv')[['sentence', 'acceptable']]
test_df = pd.read_csv('/content/RuCoLA/data/in_domain_dev.csv')[['sentence', 'acceptable']]
train, val = train_test_split(
    train_df,
    test_size=0.2,
    random_state=345,
    shuffle=True,
    stratify=train_df['acceptable']
)
train['acceptable'].mean()

np.float64(0.7451945988880063)

In [32]:
name_bert = 'ai-forever/ruRoberta-large'
tokenizer = AutoTokenizer.from_pretrained(name_bert)
model = AutoModelForSequenceClassification.from_pretrained(name_bert, num_labels=2).to(device)

train, val = Dataset.from_pandas(train), Dataset.from_pandas(val)
train = train.map(lambda x: tokenizer(x['sentence']), batched=True)
val = val.map(lambda x: tokenizer(x['sentence']), batched=True)

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    padding=True,
    return_tensors='pt'
)

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at ai-forever/ruRoberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/6295 [00:00<?, ? examples/s]

Map:   0%|          | 0/1574 [00:00<?, ? examples/s]

In [60]:
clf_metrics = evaluate.combine(['f1', 'accuracy', 'precision', 'recall'])
def compute_metrics(eval_pred):
  preds, labels = eval_pred

  return accuracy.compute(predictions=preds.argmax(axis=1), references=labels)

In [57]:
example = train_df.iloc[10:13]
example

,sentence,acceptable
10,"Не думаю, что мосты уже сняли.",1
11,"Предстояли очередные выборы, на которых он дал...",1
12,"После собрания кто-то сострил, что пока не раз...",1


In [59]:
text = example['sentence'].values.tolist()
tokens = tokenizer(text, return_tensors='pt', padding=True)

with torch.no_grad():
  y = model(**tokens).logits.argmax(axis=1)
y

tensor([0, 1, 0])

In [56]:
clf_metrics.compute(y, torch.from_numpy(example['acceptable'].values))

{'f1': 0.5,
 'accuracy': 0.3333333333333333,
 'precision': 1.0,
 'recall': 0.3333333333333333}